In [2]:
import numpy as np
from hashlib import sha512
from PIL import Image, PngImagePlugin
import os
from numpy import bitwise_xor
from matplotlib import pyplot as plt

# 生成安全的散列值 K（基于图像）
def generate_hash(image):
    """
    使用 SHA-512 算法生成 128 位的散列值 K。
    输入图像应为 2D 灰度图（numpy 数组形式）。
    """
    # 第一次哈希
    hash_object1 = sha512(image.tobytes())
    hash_hex1 = hash_object1.hexdigest()
    K1 = [int(hash_hex1[i:i + 2], 16) for i in range(0, len(hash_hex1), 2)]

    # 第二次哈希，使用第一次哈希的字节作为输入
    hash_object2 = sha512(bytes(K1))
    hash_hex2 = hash_object2.hexdigest()
    K2 = [int(hash_hex2[i:i + 2], 16) for i in range(0, len(hash_hex2), 2)]

    # 合并两个哈希结果，确保 K 长度为 128
    K = K1 + K2  # 总共 128 个元素
    return K

# 计算 ki 并初始化混沌系统（支持多个混沌系统）
def initialize_chaos_systems(K):
    """
    计算混沌系统的初始值，按新的公式修改计算方法。
    """
    k = []
    if len(K) < 12:  # 确保 K 列表有足够的元素
        raise ValueError("密钥 K 必须至少有 12 个元素。")
    
    # 对于每 12 个元素一组，执行异或并与平均值异或
    for i in range(0, len(K), 12):  # 每 12 个元素一组
        group = K[i:i+12]
        if len(group) < 12:  # 如果不足 12 个元素，则跳过
            continue
        
        # 对这一组元素进行异或
        xor_result = group[0]
        for j in range(1, len(group)):
            xor_result ^= group[j]
        
        # 计算这组元素的平均值并与异或结果异或
        avg_value = sum(group) / len(group)
        
        # 确保 xor_result 和 avg_value 都是整数进行异或操作
        ki = (xor_result ^ int(avg_value)) / 512  # 加入除以 512 的操作
        
        k.append(ki)

    return k

# 超混沌Chen系统的初始化
def super_chaotic_chen_system(x0, y0, z0, w0, steps):
    """
    离散化超混沌Chen系统。
    """
    m, n, p, q, r = 35, 3, 12, 7, 0.798  # 超混沌Chen系统的参数
    dt = 0.01  # 减小时间步长
    X, Y, Z, W = [x0], [y0], [z0], [w0]

    for _ in range(steps + 100):  # 增加额外的计算步数
        x = X[-1] + m * (Y[-1] - X[-1]) * dt
        y = Y[-1] + (-1 * X[-1] * Y[-1] + q * X[-1] + p * Y[-1]) * dt
        z = Z[-1] + (X[-1] * Y[-1] - n * Z[-1]) * dt
        w = W[-1] + (Y[-1] * Z[-1] + r * W[-1]) * dt
        X.append(x)
        Y.append(y)
        Z.append(z)
        W.append(w)

    # 跳过前 1000 步，返回剩余部分
    return X[1000:], Y[1000:], Z[1000:], W[1000:]

# 调整 Lorenz 混沌系统
def lorenz_chaotic_system(x0, y0, z0, steps):
    """
    离散化 Lorenz 混沌系统。
    """
    sigma, rho, beta = 10, 28, 8/3  # Lorenz系统参数
    dt = 0.01  # 减小时间步长
    X, Y, Z = [x0], [y0], [z0]

    for _ in range(steps + 100):  # 增加额外的计算步数
        x = X[-1] + sigma * (Y[-1] - X[-1]) * dt
        y = Y[-1] + (X[-1] * (rho - Z[-1]) - Y[-1]) * dt
        z = Z[-1] + (X[-1] * Y[-1] - beta * Z[-1]) * dt
        X.append(x)
        Y.append(y)
        Z.append(z)

    # 跳过前 1000 步，返回剩余部分
    return X[1000:], Y[1000:], Z[1000:]

def henon_chaotic_system(x0, y0, steps):
    """
    离散化 Henon 混沌系统，并添加防止溢出的处理。
    """
    a, b = 1.4, 0.3  # Henon 系统标准参数
    X, Y = [x0], [y0]

    for _ in range(steps + 100):  # 增加额外的计算步数
        x = 1 - a * X[-1]**2 + Y[-1]
        y = b * X[-1]

        # 修正溢出问题，确保 x 和 y 的值在合理范围内
        x = np.clip(x, -1, 1)  # 限制 x 的值在 [-1, 1] 范围内
        y = np.clip(y, -1, 1)  # 限制 y 的值在 [-1, 1] 范围内

        X.append(x)
        Y.append(y)

    # 跳过前 1000 步，返回剩余部分
    return X[1000:], Y[1000:]

# 组合超混沌 Chen、Lorenz 和 Henon 混沌系统
def combined_chaotic_systems(k, steps):
    """
    结合超混沌 Chen、Lorenz 和 Henon 混沌系统以生成更复杂的混沌序列。
    """
    # 确保 k 列表至少包含 10 个元素
    if len(k) < 10:
        raise ValueError("k 列表的长度必须至少为 10。")

    # 初始化超混沌 Chen 系统
    Xc, Yc, Zc, Wc = super_chaotic_chen_system(k[1], k[2], k[3], k[4], steps)
    
    # 初始化 Lorenz 系统
    Xl, Yl, Zl = lorenz_chaotic_system(k[5], k[6], k[7], steps)
    
    # 初始化 Henon 系统
    Xh, Yh = henon_chaotic_system(k[8], k[9], steps)
    
    # 组合所有系统的序列，包括 Wc 和 Zl
    combined_X = Xc + Xl + Xh
    combined_Y = Yc + Yl + Yh
    combined_Z = Zc + Zl  # 使用 Zc（超混沌 Chen 系统的 Z）和 Zl（Lorenz 系统的 Z）
    combined_W = Wc  # 包含 Wc 变量

    return combined_X, combined_Y, combined_Z, combined_W  # 返回 W 和 Z 变量

# 使用 S-盒进行扩散
def sbox_diffusion(image, s_box, M, N):
    """
    使用 S-盒对图像进行扩散。
    """
    diffused_image = np.copy(image)  # 复制图像，保证不修改原始图像
    for i in range(M):
        for j in range(N):
            idx = (i * N + j) % 256  # 修复索引计算
            diffused_image[i, j] = s_box[idx] ^ image[i, j]
    return diffused_image

# 增强混沌序列 X, Y, Z 和 W
def enhance_chaotic_sequences(X, Y, Z, W):
    """
    增强混沌序列以提高随机性，并确保结果为整数。
    包括 W 变量。
    """
    Q = [int((x + y + z + w) % 256) for x, y, z, w in zip(X, Y, Z, W)]
    return Q

# 块级操作
def block_operations(image, s_box, Q, M, N):
    """
    对图像进行块操作以扩散像素。
    """
    T = np.zeros((M, N), dtype=int)
    for i in range(M):
        for j in range(N):
            idx = int(Q[(i * N + j) % len(Q)])  # 修复索引计算
            T[i, j] = s_box[idx] ^ image[i, j]
    return T

# 逆向 S-盒扩散
def inverse_sbox_diffusion(image, s_box, M, N):
    """
    逆向扩散操作，使用 S-盒还原像素值。
    """
    diffused_image = np.copy(image)
    for i in range(M):
        for j in range(N):
            idx = (i * N + j) % 256
            diffused_image[i, j] = s_box[idx] ^ image[i, j]
    return diffused_image

# 逆向块级操作
def inverse_block_operations(image, s_box, Q, M, N):
    """
    逆向块级扩散操作。
    """
    T = np.zeros((M, N), dtype=int)
    for i in range(M):
        for j in range(N):
            idx = int(Q[(i * N + j) % len(Q)])  # 修复索引计算
            T[i, j] = s_box[idx] ^ image[i, j]
    return T
def arnold_transform(image, iterations):
        """
        对图像进行Arnold变换（置乱）。
        :param image: 输入图像（灰度图或彩色图）
        :param iterations: 迭代次数
        :return: 置乱后的图像
        """
        rows, cols = image.shape[:2]
        transformed_image = image.copy()

        for _ in range(iterations):
            new_image = np.zeros_like(transformed_image)
            for i in range(rows):
                for j in range(cols):
                    new_i = (i + j) % rows
                    new_j = (i + 2 * j) % cols
                    new_image[new_i, new_j] = transformed_image[i, j]
            transformed_image = new_image

        return transformed_image

def inverse_arnold_transform(image, iterations):
        """
        对图像进行逆Arnold变换（恢复）。
        :param image: 输入图像（灰度图或彩色图）
        :param iterations: 迭代次数
        :return: 恢复后的图像
        """
        rows, cols = image.shape[:2]
        transformed_image = image.copy()

        for _ in range(iterations):
            new_image = np.zeros_like(transformed_image)
            for i in range(rows):
                for j in range(cols):
                    new_i = (2 * i - j) % rows
                    new_j = (-i + j) % cols
                    new_image[new_i, new_j] = transformed_image[i, j]
            transformed_image = new_image

        return transformed_image
# 加密图像
def encrypt_image(image, s_box, M, N):
    """
    使用混沌加密算法加密灰度图像。
    返回加密后的图像和密钥 K。
    """
    K = generate_hash(image)
    print(f"[加密] 生成的密钥 K 长度: {len(K)}")  # 应为 128
    k = initialize_chaos_systems(K)
    X, Y, Z, W = combined_chaotic_systems(k, M * N // 4)  # 获取 W 和 Z 变量
    Q = enhance_chaotic_sequences(X, Y, Z, W)  # 增强时传递 W
    diffused_image = sbox_diffusion(image, s_box, M, N)
    encrypted_image = block_operations(diffused_image, s_box, Q, M, N)
    return np.clip(encrypted_image, 0, 255), K

# 解密图像
def decrypt_image(image, s_box, M, N, K):
    """
    使用混沌加密算法解密灰度图像。
    """
    k = initialize_chaos_systems(K)
    X, Y, Z, W = combined_chaotic_systems(k, M * N // 4)  # 获取 W 和 Z 变量
    Q = enhance_chaotic_sequences(X, Y, Z, W)  # 增强时传递 W
    # 逆向块级扩散
    block_reversed_image = inverse_block_operations(image, s_box, Q, M, N)
    # 逆向 S-盒扩散
    decrypted_image = inverse_sbox_diffusion(block_reversed_image, s_box, M, N)
    return np.clip(decrypted_image, 0, 255)  # 确保像素值范围正确

# 序列化和反序列化 K
def serialize_K(K):
    """
    将 K 列表序列化为字符串，用于存储在元数据中。
    """
    return ','.join(map(str, K))

def deserialize_K(k_string):
    """
    将序列化的 K 字符串反序列化为列表。
    """
    return list(map(int, k_string.split(',')))

# 生成img_key（确定性）
def generate_img_key(K, M, N):
    """
    基于密钥K生成确定性的img_key。
    """
    # 使用序列化K的SHA-512哈希值来设定种子
    K_bytes = bytes(K)
    seed = int.from_bytes(sha512(K_bytes).digest()[:8], 'big')  # 从哈希中提取种子
    rng = np.random.default_rng(seed)
    return rng.integers(0, 256, size=(M, N), dtype=np.uint8)

# 主函数示例用法
def main():
    # S-盒定义（保持不变）
    s_box = [
        30, 43, 217, 136, 160, 233, 123, 179, 111, 93, 184, 114, 241, 4, 149, 107,
        251, 21, 226, 131, 99, 25, 6, 48, 180, 186, 61, 7, 134, 191, 249, 109,
        177, 23, 198, 105, 78, 183, 141, 89, 253, 68, 192, 90, 60, 49, 39, 240,
        101, 225, 98, 63, 210, 77, 207, 120, 124, 129, 147, 162, 176, 11, 250, 59,
        148, 37, 199, 9, 92, 34, 248, 53, 24, 62, 151, 238, 138, 82, 224, 26,
        178, 44, 29, 206, 95, 132, 18, 128, 33, 185, 212, 236, 223, 169, 71, 170,
        254, 219, 188, 156, 14, 182, 211, 31, 86, 166, 234, 244, 171, 116, 3, 252,
        5, 193, 140, 13, 133, 126, 110, 35, 103, 139, 201, 173, 66, 167, 209, 158,
        117, 70, 222, 119, 175, 174, 200, 81, 17, 146, 46, 232, 67, 87, 214,
        230, 51, 181, 32, 115, 135, 55, 72, 157, 122, 45, 22, 10, 79, 15, 50,
        187, 102, 83, 41, 216, 194, 229, 88, 12, 227, 57, 125, 19, 54, 190,
        127, 94, 220, 197, 144, 242, 85, 208, 73, 189, 163, 16, 152, 243, 239, 96,
        106, 36, 64, 245, 161, 47, 97, 150, 113, 213, 56, 130, 100, 27, 165, 215,
        195, 143, 20, 58, 108, 52, 74, 218, 69, 2, 153, 145, 1, 38, 205, 203,
        65, 235, 204, 247, 172, 80, 121, 28, 202, 246, 142, 228, 0, 91, 8, 104,
        221, 155, 255, 75, 137, 40, 168, 118, 154, 196, 42, 76, 159, 84, 164, 231,
        112, 237
    ]

    # 示例路径设置
    encryption_input_path = r"OIP-C.jpg"  # 原始图像路径
    encryption_output_path = r"8.png"  # 加密后图像路径
    decryption_input_path = encryption_output_path  # 加密图像路径（用于解密）
    decryption_output_path = r"88.png"  # 解密后图像路径

    # 加密过程
    print("开始加密过程...")
    try:
        loaded_image = Image.open(encryption_input_path).convert('L')
    except Exception as e:
        print(f"无法加载图像: {e}")
        return
    image_array = np.array(loaded_image)
    M, N = image_array.shape
    # 置乱图像
    iterations = 20  # 迭代次数
    image_array = arnold_transform(image_array, iterations)
    encrypted_image, K = encrypt_image(image_array, s_box, M, N)
    k_serialized = serialize_K(K)
    metadata = PngImagePlugin.PngInfo()
    metadata.add_text("K", k_serialized)

    Q_jiami = np.zeros([M, N], dtype=np.float32)
    img_key = generate_img_key(K, M, N)
    for i in range(M):
        for j in range(N):
            Q_jiami[i, j] = bitwise_xor(encrypted_image[i, j], img_key[i, j])
    Q_jiami[1:64, 1:128] = 255
    encrypted_pil_image = Image.fromarray(Q_jiami.astype(np.uint8))
    encrypted_pil_image.save(encryption_output_path, pnginfo=metadata)
    print(f"加密后的图像已保存到：{encryption_output_path}")

    # 解密过程
    print("开始解密过程...")
    try:
        loaded_encrypted_image = Image.open(decryption_input_path)
    except Exception as e:
        print(f"无法加载加密图像: {e}")
        return

    k_serialized = loaded_encrypted_image.info.get("K")
    if k_serialized is None:
        print("加密图像中未找到密钥 K 的元数据。解密失败。")
        return
    K_decrypt = deserialize_K(k_serialized)

    grayscale_encrypted_image = loaded_encrypted_image.convert('L')
    decrypted_image_array = np.array(grayscale_encrypted_image)
    M_dec, N_dec = decrypted_image_array.shape
    Q_jiemi = np.zeros([M_dec, N_dec], dtype=np.float32)
    for i in range(M_dec):
        for j in range(N_dec):
            Q_jiemi[i, j] = bitwise_xor(decrypted_image_array[i, j], img_key[i, j])
    decrypted_image = decrypt_image(Q_jiemi.astype(np.uint8), s_box, M_dec, N_dec, K_decrypt)
    decrypted_image = inverse_arnold_transform(decrypted_image, iterations)
    decrypted_pil_image = Image.fromarray(decrypted_image.astype(np.uint8))
    #
    decrypted_pil_image.save(decryption_output_path)
    print(f"解密后的图像已保存到：{decryption_output_path}")



if __name__ == "__main__":
    main()



开始加密过程...
[加密] 生成的密钥 K 长度: 128
加密后的图像已保存到：8.png
开始解密过程...
解密后的图像已保存到：88.png
